# Модуль 4 · Урок 1 — Django REST Framework: основи, моделі, serializers

👋 Ви вже знаєте Python, ООП і бази даних. Час зібрати це в **справжній бекенд** — сервіс, до
якого звертаються мобільні застосунки, фронтенд та інші сервіси. Робитимемо це на
**Django + Django REST Framework (DRF)** — найпопулярнішому стеку для REST API на Python.

**Передумова:** [Модуль 2 (ООП)](../modul-2-oop/lektsiya-1-osnovy-oop.ipynb) та
[Модуль 3 (Бази даних)](../modul-3-bazy-danykh/lektsiya-1-sql-osnovy.ipynb) — DRF стоїть «на плечах» класів і БД.

### Що ви вмітимете після уроку
- пояснити, що таке **REST** і за що відповідає кожен **HTTP-метод**;
- розуміти **архітектуру Django + DRF** і структуру бекенд-проєкту;
- створити **модель** й зробити **міграції**;
- написати **serializer** (перетворення об'єкт ⇄ JSON) з валідацією;
- зробити перший ендпоінт через **APIView** та **Generic Views**.

> 📌 **Як читати цей зошит.** DRF-код запускається у **справжньому проєкті** (не в цьому файлі),
> тож приклади подано як блоки коду з реалістичним результатом. Наприкінці уроку — інструкція,
> як підняти проєкт локально й повторити все руками.
> Позначка **🔎 Важливо знати** — те, що майже завжди питають на співбесіді.

## 1. Що таке REST і HTTP-методи

**API** (Application Programming Interface) — «розетка», через яку одна програма звертається до
іншої. **REST API** — найпоширеніший стиль web-API, побудований поверх **HTTP**.

Ключова ідея REST: усе — це **ресурси** (наприклад, «стаття», «користувач»), у кожного свій
**URL**, а **дія** над ресурсом визначається **HTTP-методом**:

| Метод | Дія (CRUD) | Приклад | Ідемпотентний? |
|-------|-----------|---------|----------------|
| `GET` | Read — прочитати | `GET /articles/` (список), `GET /articles/5/` (один) | ✅ так |
| `POST` | Create — створити | `POST /articles/` (тіло — нова стаття) | ❌ ні |
| `PUT` | Update — замінити **повністю** | `PUT /articles/5/` (усі поля) | ✅ так |
| `PATCH` | Update — змінити **частково** | `PATCH /articles/5/` (лише деякі поля) | ❌ (зазвичай) |
| `DELETE` | Delete — видалити | `DELETE /articles/5/` | ✅ так |

**CRUD** = Create / Read / Update / Delete — чотири базові операції над даними. Саме їх і покриває REST.

### Коди статусу відповіді (status codes)
Сервер на кожен запит повертає **код статусу** — трицифрове число, що каже, чим усе скінчилось:

| Група | Значення | Часті приклади |
|-------|----------|----------------|
| **2xx** | успіх | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | перенаправлення | `301 Moved Permanently`, `304 Not Modified` |
| **4xx** | помилка клієнта | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found` |
| **5xx** | помилка сервера | `500 Internal Server Error`, `502 Bad Gateway` |

In [ ]:
# Маленька шпаргалка як звичайні Python-структури (це наочність, ще не DRF):
http_methods = {
    "GET":    "прочитати ресурс (список або один)",
    "POST":   "створити новий ресурс",
    "PUT":    "замінити ресурс повністю",
    "PATCH":  "змінити ресурс частково",
    "DELETE": "видалити ресурс",
}
for method, meaning in http_methods.items():
    print(f"{method:6} -> {meaning}")

print()
status = {200: "OK", 201: "Created", 204: "No Content",
          400: "Bad Request", 401: "Unauthorized", 403: "Forbidden",
          404: "Not Found", 500: "Internal Server Error"}
for code, name in status.items():
    group = {2: "успіх", 4: "помилка клієнта", 5: "помилка сервера"}[code // 100]
    print(f"{code} {name:22} ({group})")

> 🔎 **Важливо знати — принципи REST.** REST — це не бібліотека, а **набір домовленостей**:
> **1) ресурси й URL** (іменники, не дієслова: `/articles/`, а не `/getArticles`);
> **2) HTTP-метод визначає дію** (не роби «видалення» через GET);
> **3) stateless** — кожен запит самодостатній, сервер не зберігає «сесію кроків» між запитами
> (тому й токен/креденшали шлють у кожному запиті);
> **4) єдиний формат** — зазвичай **JSON**;
> **5) правильні status codes**. Дотримання цих правил і робить API «RESTful».

## 2. Архітектура Django + DRF

**Django** — «важкий» веб-фреймворк: ORM (робота з БД через класи), маршрутизація, адмінка,
міграції. **Django REST Framework (DRF)** — надбудова над Django, що перетворює його на зручний
інструмент для **REST API** (serializers, ViewSets, автентифікація, пагінація тощо).

Django будується за патерном **MVT** (Model–View–Template) — це варіація класичного **MVC**:

| Django (MVT) | Класичний MVC | Роль |
|--------------|---------------|------|
| **Model** | Model | дані + робота з БД (ORM) |
| **View** | Controller | логіка обробки запиту |
| **Template / Serializer** | View | що віддати клієнту (HTML або JSON) |

У REST-API замість HTML-шаблонів віддаємо **JSON**, і цю роль бере на себе **serializer**.

### Шлях запиту крізь DRF
```
        HTTP-запит  (GET /api/articles/5/)
              │
              ▼
   ┌─────────────────────┐
   │  urls.py (Router)   │  зіставляє URL із потрібним view
   └─────────┬───────────┘
             ▼
   ┌─────────────────────┐
   │   View / ViewSet    │  логіка: що робити із запитом
   └─────────┬───────────┘
             ▼
   ┌─────────────────────┐
   │     Serializer      │  об'єкт ⇄ JSON (+ валідація вхідних даних)
   └─────────┬───────────┘
             ▼
   ┌─────────────────────┐
   │     Model (ORM)     │  читає/пише в базу даних
   └─────────┬───────────┘
             ▼
         База даних
             │
             ▼
        HTTP-відповідь (JSON + status code)
```

## 3. Структура бекенд-проєкту

У Django розрізняють **проєкт** (увесь сайт, налаштування) і **застосунки** (`app` — окремі
функціональні частини: `articles`, `users`, `orders`). Це і є **розділення відповідальностей**
з Модуля 2 — кожен app відповідає за свою частину.

```
myproject/
├── manage.py              # CLI: запуск сервера, міграції, команди
├── myproject/             # ПАКЕТ ПРОЄКТУ (налаштування всього сайту)
│   ├── settings.py        # конфігурація (БД, INSTALLED_APPS, DRF)
│   ├── urls.py            # головна таблиця маршрутів
│   └── wsgi.py / asgi.py  # точки входу для сервера
└── articles/              # ЗАСТОСУНОК (одна функціональна частина)
    ├── models.py          # моделі (таблиці БД)
    ├── serializers.py     # serializers (об'єкт ⇄ JSON)   ← додаємо самі
    ├── views.py           # views / viewsets (логіка ендпоінтів)
    ├── urls.py            # маршрути цього app            ← додаємо самі
    ├── admin.py           # реєстрація в адмінці
    └── migrations/        # історія змін схеми БД
```

> 💡 `serializers.py` та `urls.py` всередині app Django **не створює автоматично** — їх додають
> вручну. Решту файлів генерує команда `startapp`.

## 4. Встановлення та налаштування DRF

Усе робимо в терміналі, у **віртуальному оточенні** (щоб залежності проєкту не мішалися із
системними):

```bash
# 1. Віртуальне оточення
python -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate

# 2. Встановлення Django та DRF
pip install django djangorestframework

# 3. Створення проєкту та застосунку
django-admin startproject myproject .
python manage.py startapp articles
```

Далі підключаємо DRF і наш app у `myproject/settings.py`:

```python
# myproject/settings.py
INSTALLED_APPS = [
    "django.contrib.admin",
    "django.contrib.auth",
    "django.contrib.contenttypes",
    "django.contrib.sessions",
    "django.contrib.messages",
    "django.contrib.staticfiles",
    "rest_framework",        # ← DRF
    "articles",              # ← наш застосунок
]
```

```bash
# Застосувати початкові міграції та підняти сервер
python manage.py migrate
python manage.py runserver      # http://127.0.0.1:8000/
```

## 5. Моделі та міграції

**Модель** — це Python-клас, що описує таблицю в БД (згадайте Модуль 3). Django сам створює
таблицю за класом — SQL руками писати не треба.

```python
# articles/models.py
from django.db import models

class Article(models.Model):
    title = models.CharField(max_length=200)              # VARCHAR(200)
    content = models.TextField()                          # TEXT
    is_published = models.BooleanField(default=False)     # BOOLEAN
    created_at = models.DateTimeField(auto_now_add=True)  # ставиться раз при створенні

    def __str__(self):                                    # як показувати об'єкт (Модуль 2!)
        return self.title
```

Після зміни моделей — **дві команди міграцій**:

```bash
python manage.py makemigrations   # згенерувати файл міграції за змінами в models.py
python manage.py migrate          # застосувати їх до БД (створити/змінити таблиці)
```

> 🔎 **Важливо знати — навіщо міграції?** Міграція — це **версійована зміна схеми БД** (файл у
> `migrations/`). Вони дають командам синхронну структуру таблиць, історію змін і можливість
> відкату. `makemigrations` — *описує* зміну; `migrate` — *застосовує*. Це «git для схеми БД».

## 6. Serializers — серце DRF

**Serializer** робить дві речі:
1. **Серіалізація:** об'єкт Python / рядок моделі → JSON (щоб віддати клієнту).
2. **Десеріалізація + валідація:** JSON із запиту → перевірені дані → об'єкт (щоб зберегти).

Спершу — інтуїція на чистому Python (без DRF): «серіалізувати» — це перетворити об'єкт на
структуру, яку можна віддати як JSON.

In [ ]:
import json

# Уявімо об'єкт (у реальності — рядок із БД). Серіалізація = перетворити його на JSON:
article = {"id": 1, "title": "Привіт, DRF", "is_published": True}
json_text = json.dumps(article, ensure_ascii=False)
print("Серіалізація (об'єкт -> JSON):")
print(json_text)

# Десеріалізація = розібрати JSON назад у Python-об'єкт:
data = json.loads(json_text)
print("\nДесеріалізація (JSON -> об'єкт):")
print(data["title"], "| published:", data["is_published"])

Саме це й робить DRF, тільки розумніше — з валідацією й прив'язкою до моделі. Найчастіше беруть
**`ModelSerializer`**: він сам будує поля за моделлю.

```python
# articles/serializers.py
from rest_framework import serializers
from .models import Article

class ArticleSerializer(serializers.ModelSerializer):
    class Meta:
        model = Article
        fields = ["id", "title", "content", "is_published", "created_at"]
        read_only_fields = ["id", "created_at"]   # ці поля не приймаємо від клієнта
```

### Валідація
DRF перевіряє типи й обмеження полів автоматично. Свою перевірку додають методами
`validate_<поле>` (одне поле) та `validate` (кілька полів разом):

```python
class ArticleSerializer(serializers.ModelSerializer):
    class Meta:
        model = Article
        fields = ["id", "title", "content", "is_published", "created_at"]

    def validate_title(self, value):              # валідація ОДНОГО поля
        if len(value) < 5:
            raise serializers.ValidationError("Заголовок закороткий (мін. 5 символів).")
        return value

    def validate(self, data):                     # валідація КІЛЬКОХ полів разом
        if data.get("is_published") and not data.get("content"):
            raise serializers.ValidationError("Не можна публікувати статтю без тексту.")
        return data
```

Якщо дані некоректні — DRF сам поверне `400 Bad Request` з описом помилки, наприклад:
```json
{ "title": ["Заголовок закороткий (мін. 5 символів)."] }
```

> 🎯 **Питання: `Serializer` vs `ModelSerializer`?** `Serializer` — базовий, поля описуєте
> вручну (гнучко, коли дані не збігаються з моделлю). `ModelSerializer` — будує поля **за
> моделлю** автоматично (`Meta.model` + `fields`) і дає готові `.create()`/`.update()`. Для
> CRUD над моделлю майже завжди беруть `ModelSerializer`.

## 7. APIView — перший ендпоінт

**`APIView`** — базовий клас DRF: ви самі описуєте методи `get`, `post`, `put`, `delete`.
Максимум контролю, але й найбільше коду. Приклад — список статей і створення:

```python
# articles/views.py
from rest_framework.views import APIView
from rest_framework.response import Response
from rest_framework import status
from .models import Article
from .serializers import ArticleSerializer

class ArticleListAPIView(APIView):
    def get(self, request):                       # GET /api/articles/
        articles = Article.objects.all()
        serializer = ArticleSerializer(articles, many=True)   # many=True — список
        return Response(serializer.data)          # 200 OK + JSON

    def post(self, request):                      # POST /api/articles/
        serializer = ArticleSerializer(data=request.data)
        if serializer.is_valid():                 # запускає валідацію
            serializer.save()                     # створює рядок у БД
            return Response(serializer.data, status=status.HTTP_201_CREATED)
        return Response(serializer.errors, status=status.HTTP_400_BAD_REQUEST)
```

Маршрути:
```python
# articles/urls.py
from django.urls import path
from .views import ArticleListAPIView

urlpatterns = [
    path("articles/", ArticleListAPIView.as_view()),
]
```
```python
# myproject/urls.py
from django.urls import path, include

urlpatterns = [
    path("api/", include("articles.urls")),   # усі маршрути app під /api/
]
```

Тепер `GET http://127.0.0.1:8000/api/articles/` поверне JSON-список, наприклад:
```json
[
  {"id": 1, "title": "Привіт, DRF", "content": "...", "is_published": true, "created_at": "2026-07-10T09:00:00Z"}
]
```

## 8. Generic Views — менше коду

Той самий CRUD DRF уміє робити майже без коду через **generic views** — готові класи під типові
задачі. Ви лише вказуєте `queryset` і `serializer_class`:

```python
# articles/views.py
from rest_framework import generics
from .models import Article
from .serializers import ArticleSerializer

# GET (список) + POST (створення) — одразу:
class ArticleListCreate(generics.ListCreateAPIView):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer

# GET (один) + PUT/PATCH (оновлення) + DELETE — одразу:
class ArticleDetail(generics.RetrieveUpdateDestroyAPIView):
    queryset = Article.objects.all()
    serializer_class = ArticleSerializer
```
```python
# articles/urls.py
from django.urls import path
from .views import ArticleListCreate, ArticleDetail

urlpatterns = [
    path("articles/", ArticleListCreate.as_view()),        # /api/articles/
    path("articles/<int:pk>/", ArticleDetail.as_view()),   # /api/articles/5/
]
```

Порівняйте обсяг коду: в `APIView` ви пишете кожен метод; у `ListCreateAPIView` усе вже
всередині. Ось найкорисніші generic-класи:

| Клас | HTTP-методи | Що робить |
|------|-------------|-----------|
| `ListAPIView` | GET | список |
| `CreateAPIView` | POST | створення |
| `RetrieveAPIView` | GET | один об'єкт |
| `ListCreateAPIView` | GET, POST | список + створення |
| `RetrieveUpdateDestroyAPIView` | GET, PUT, PATCH, DELETE | читання + оновлення + видалення |

> 🔎 **Ієрархія:** `APIView` → generic views → (наступний урок) `ViewSets`. Що вище — то більше
> контролю й коду; що нижче — то менше коду й більше «магії». Junior має розуміти **обидва
> кінці**: generic — для швидкості, `APIView` — коли треба нестандартна логіка.

## 9. 🎯 Що спитають на співбесіді (Урок 1)

| Питання | Коротка відповідь |
|---------|-------------------|
| Що таке REST? | Стиль API поверх HTTP: ресурси+URL, дія = HTTP-метод, stateless, JSON |
| `PUT` vs `PATCH`? | `PUT` — заміна **повністю**; `PATCH` — **часткове** оновлення |
| Який метод НЕ ідемпотентний? | `POST` (два виклики = два об'єкти); `GET/PUT/DELETE` — ідемпотентні |
| `200` vs `201` vs `204`? | OK / створено (Create) / успіх без тіла (напр. DELETE) |
| `401` vs `403`? | Не автентифікований (хто ти?) vs немає прав (тебе знаю, але не можна) |
| Що робить serializer? | Об'єкт ⇄ JSON + валідація вхідних даних |
| `Serializer` vs `ModelSerializer`? | Ручні поля vs авто-поля за моделлю |
| Навіщо міграції? | Версійована зміна схеми БД (`makemigrations` описує, `migrate` застосовує) |
| `APIView` vs generic views? | Повний контроль/більше коду vs готові класи під CRUD |

# ✅ Підсумок уроку
- **REST** — ресурси+URL, дія = **HTTP-метод** (`GET/POST/PUT/PATCH/DELETE`), stateless, JSON, правильні status codes.
- **Django (MVT) + DRF**: Model (ORM) → View/ViewSet → Serializer (⇄ JSON) → БД.
- **Проєкт** складається з **app-ів** (розділення відповідальностей); `serializers.py`/`urls.py` в app додаємо самі.
- **Модель** описує таблицю; зміни застосовуються через `makemigrations` + `migrate`.
- **Serializer** (найчастіше `ModelSerializer`) перетворює об'єкт ⇄ JSON і **валідує** (`validate_<поле>`, `validate`).
- **Ендпоінти:** `APIView` (усе руками) або **generic views** (`ListCreateAPIView`, `RetrieveUpdateDestroyAPIView`) — швидкий CRUD.

### 🧪 Спробувати руками
Повний робочий мінімум (щоб повторити все вище):
```bash
python -m venv venv && source venv/bin/activate
pip install django djangorestframework
django-admin startproject myproject .
python manage.py startapp articles
# -> додати 'rest_framework' і 'articles' у INSTALLED_APPS
# -> написати models.py, serializers.py, views.py, urls.py (як у прикладах)
python manage.py makemigrations && python manage.py migrate
python manage.py runserver
```

### 📝 Домашнє завдання
[domashnie-zavdannia-lektsiya-1.ipynb](domashnie-zavdannia-lektsiya-1.ipynb)

### ▶️ Далі
Урок 2 — **ViewSets, Routers, фільтрація/пошук/сортування, пагінація, автентифікація (Token),
permissions та обробка помилок**.

### 📚 Хочу знати більше
- DRF (офіційно, quickstart): <https://www.django-rest-framework.org/tutorial/quickstart/>
- Django (офіційно): <https://docs.djangoproject.com/>
- REST — пояснення: <https://restfulapi.net/>
- HTTP status codes (MDN): <https://developer.mozilla.org/en-US/docs/Web/HTTP/Status>